In [1]:
import os
import sys
import argparse
import pickle
from collections import defaultdict
from itertools import combinations
from functools import partial
from tqdm import tqdm
import math
import time
import gzip

import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse
from scipy import stats
from scipy.stats import pearsonr

In [2]:
from statsmodels.stats.multitest import multipletests
import numba as nb

In [134]:
%load_ext autoreload
%autoreload 2

from make_hubs import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
pd.set_option("display.max_columns", 100)

### Step 1: age bins

In [3]:
AGE_BINS = np.array([10, 21, 31, 41, 51, 61, 71])

In [109]:
adata_full.obs.isna().sum()

GSE_id                0
GSM_id                0
Age                1498
Tissue_global         0
cell_type_final       0
dtype: int64

In [ ]:
adata_full = adata_full[~adata_full.obs.Age.isna()].copy()  # nan from heart
adata_full

AnnData object with n_obs × n_vars = 1454732 × 42947
    obs: 'GSE_id', 'GSM_id', 'Age', 'Tissue_global', 'cell_type_final'
    uns: 'Tissue_global_colors'
    obsm: 'X_scANVI', 'X_scVI', 'X_umap', '_scvi_extra_categorical_covs'

In [ ]:
# bins age in 10 year bins
def age_bin(x):
    """return lower age bin bound"""
    for upper_bound in range(21, 80, 10):
        if x < 21:
            return 10
        elif upper_bound > x:
            return upper_bound - 10
    # for the oldest:
    return upper_bound

In [125]:
adata = adata_full.copy()
del adata_full

In [ ]:
adata.obs["Age_bin"] = adata.obs.Age.map(age_bin)

In [ ]:
# run script

### Step 2: identify age-dynamic genes

In [ ]:
adata = sc.read_h5ad("./data/adata_ribo_downsampled.h5ad")
adata

AnnData object with n_obs × n_vars = 1454732 × 42947
    obs: 'GSE_id', 'GSM_id', 'Age', 'Tissue_global', 'cell_type_final'
    uns: 'Tissue_global_colors'
    obsm: 'X_scANVI', 'X_scVI', 'X_umap', '_scvi_extra_categorical_covs'

In [ ]:
print(
    "adata.X: min =",
    adata.X.min(),
    ", max =",
    adata.X.max(),
    ", mean ≈",
    adata.X.mean(),
)
if adata.raw:
    print(
        "adata.raw.X: min =",
        adata.raw.X.min(),
        ", max =",
        adata.raw.X.max(),
        ", mean ≈",
        adata.raw.X.mean(),
    )

for key in adata.layers:
    layer = adata.layers[key]
    print(f"{key}: min = {layer.min()}, max = {layer.max()}, mean ≈ {layer.mean()}")
print("\n")

adata.X: min = 0.0 , max = 55441.0 , mean ≈ 0.10928421




In [ ]:
tissue_cell_combinations = [
    (tissue, cell_type)
    for tissue in adata.obs["Tissue_global"].unique()
    for cell_type in adata.obs[adata.obs["Tissue_global"] == tissue]["cell_type_final"]
    .unique()
    .sort_values()
]

#### read permutations , pval, filt, fdr

In [ ]:
adt = sc.read_h5ad(
    "./data/senepy_denovo_signatures_code/6.3_adata_ribo_downsampled.h5ad"
)

In [ ]:
def calculate_p_values(null_df, test_values):
    """Вычисляет эмпирические p-values для каждого гена.
    Args:
        null_df (pd.DataFrame): строки = гены, столбцы = пермутации. Исключить dtype=str
        test_values (pd.Series | array-like) наблюдаемая метрика для каждого гена.
    Returns:
        pd.Series: p-values.
    """
    null_vals = null_df.to_numpy()
    test_vals = test_values.to_numpy()

    N = null_vals.shape[1]  # n cells

    counts = (null_vals > test_vals[:, None]).sum(axis=1)
    p_values = (counts + 1) / (N + 1)

    return p_values


def extra_filt(x):
    """
    фильтрует гены с большим одиночным всплеском (если максимум в 2 раза больше, чем значение старшего бина)
    OR
    гены с «ранним пиком»
    """
    x = x[x >= 0]
    if x.max() > 2 * x.iloc[-1] or int(x.idxmax()) <= 41:
        return False
    else:
        return True


def origi_filt(df):
    cf = df.copy()

    cf = cf[
        (cf["lin.slope"] > 0)
        &
        # положительный тренд
        (cf["max_prop"] < 0.2)
        &
        # не слишком повсеместно экспрессируемый ген - raw 0.2
        (cf["young_prop"] < 0.02)
        &
        # низкая экспрессия в молодых бинах - raw .05
        (cf["old_vs_young_delta"] > 0.025)
    ]
    # существенный абсолютный прирост - raw .015

    cf = cf[cf[AGE_BINS.astype(str)].apply(extra_filt, axis=1)]

    return df.gene.isin(cf.gene)

In [ ]:
res = []
for tissue, ct in tqdm(
    tissue_cell_combinations, total=len(tissue_cell_combinations), desc="Tissue.Cell"
):
    if ct == "Unlabeled":
        continue
    try:
        test_df = pd.read_csv(
            f"./data/sp_downsampled_perm_out/{tissue}.{ct}.main_metrics.csv"
        )
        null_df = pd.read_csv(
            f"./data/sp_downsampled_perm_out/{tissue}.{ct}.permutations_GAD.csv"
        )
    except:
        continue

    # отсортировать по генам в одинаковом порядке - мб избыточно, ибо они и так дб в одинаковом порядке
    null_df = null_df.set_index("gene").loc[test_df["gene"]].reset_index()

    test_df["pval"] = calculate_p_values(null_df.iloc[:, 1:], test_df["GAD"])
    test_df["origFilt"] = origi_filt(test_df)
    test_df.insert(1, column="tissue", value=tissue)
    test_df.insert(2, column="cell", value=ct)
    test_df["tissue.cell"] = f"{tissue}.{ct}"

    ######### FDR correction #########
    pvals = test_df["pval"].to_numpy()
    mask = ~np.isnan(pvals)  # обрабатывать NaN тут не особо надо, но пусть будет
    # rej = np.full(len(pvals), False) # отклонена ли H0
    qvals = np.full(len(pvals), np.nan)

    reject, pvals_corr, _, _ = multipletests(pvals[mask], alpha=0.05, method="fdr_bh")
    # rej[mask] = reject
    qvals[mask] = pvals_corr

    test_df["q_bh"] = qvals
    # test_df['significant_bh'] = rej

    res.append(test_df)


Tissue.Cell: 100%|████████████████████████████████████████████████████████████████████| 130/130 [06:11<00:00,  2.86s/it]


In [9]:
tc_df = pd.concat(res, axis=0)

In [ ]:
tc_df.to_csv("./data/all_tc_GAD_stats_downsampled.csv")

### Step 3: paired correlations of GAD genes

In [ ]:
# run pearson corr & perm script

In [ ]:
n = pd.read_csv(
    "./data/senepy_denovo_signatures_code/sp_pearson_perm_output/Colon.CD4T.pearson_perm_stats.csv"
)
n

,tissue,cell,gene1,gene2,r,p,rnd_r_mean,rnd_p_mean,rnd_r_q99,rnd_p_q01,rnd_r_sd,rnd_p_sd
0,Colon,CD4T,GNLY,PPARG,-0.007809,0.615105,-8.607838e-06,0.521336,0.040441,0.009190,0.015088,0.300820
1,Colon,CD4T,GNLY,FAM153CP,0.004242,0.784776,1.310019e-04,0.492335,0.034029,0.028409,0.015134,0.266464
2,Colon,CD4T,GNLY,PCP4,-0.006974,0.653422,-2.050895e-04,0.548955,0.064275,0.000034,0.015209,0.226388
3,Colon,CD4T,GNLY,ABCC3,0.020035,0.197014,5.701708e-04,0.496240,0.054548,0.000440,0.016347,0.291840
4,Colon,CD4T,GNLY,CLCA1,0.000592,0.969571,1.299956e-03,0.502611,0.037922,0.014707,0.016253,0.316892
...,...,...,...,...,...,...,...,...,...,...,...,...
166,Colon,CD4T,ADGRG7,KRT8,0.014128,0.362983,-1.926639e-03,0.500715,0.038126,0.014062,0.015154,0.285279
167,Colon,CD4T,ADGRG7,EHF,0.032530,0.036168,-2.246046e-07,0.586471,0.032918,0.035807,0.015336,0.234251
168,Colon,CD4T,EDNRB-AS1,KRT8,-0.013132,0.397817,7.835077e-04,0.465923,0.052403,0.000735,0.017019,0.288573
169,Colon,CD4T,EDNRB-AS1,EHF,-0.006917,0.656048,-6.208488e-05,0.550163,0.063755,0.000040,0.015351,0.227912


`r` 	реальная корреляция Пирсона

`p` 	реальное p-value

`rnd_r_mean`    	средняя корреляция после случайных перестановок

`rnd_p_mean`    	среднее p-value после случайных перестановок

`rnd_r_q99` 	99-й перцентиль случайных корреляций

`rnd_p_q01` 	1-й перцентиль случайных p-value

`rnd_r_sd`  	стандартное отклонение случайных корреляций

`rnd_p_sd`  	стандартное отклонение случайных p-value
